# Gemma 2 Evaluation Inference
Generates therapist responses for all 181 eval scenarios using:
- **base**: Gemma 2 2B IT (no fine-tuning)
- **v1**: Fine-tuned v1 (95 examples)
- **v2**: Fine-tuned v2 (363 examples)

Outputs saved to Google Drive → `data/evaluation/model_responses_{config}.jsonl`

**Run all cells top to bottom. Each config section is independent.**

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# Run this cell, then: Runtime > Restart session, then skip this cell and run from Cell 2
# ============================================================

!pip uninstall -y numpy scipy scikit-learn -q
!pip install -q numpy==1.26.4 scipy==1.13.1 scikit-learn==1.5.2
!pip install -q bitsandbytes
!pip install -q peft>=0.12.0 accelerate>=0.33.0 transformers>=4.44.0 trl>=1.0.0

print('\nDone! Now: Runtime > Restart session — then run from Cell 2 (skip this cell)')

In [ ]:
# ============================================================
# CELL 2: Mount Drive + Load keys (after runtime restart)
# ============================================================
import os, json
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_DIR = '/content/drive/MyDrive/mi-therapy-capstone'
KEYS_FILE = f'{PROJECT_DIR}/api_keys.json'

with open(KEYS_FILE) as f:
    keys = json.load(f)

from huggingface_hub import login
login(token=keys['hf_token'], add_to_git_credential=False)
print(f'Logged into HuggingFace as: {keys["hf_username"]}')

# Paths
EVAL_SCENARIOS  = f'{PROJECT_DIR}/data/evaluation/eval_scenarios.jsonl'
OUTPUT_DIR      = f'{PROJECT_DIR}/data/evaluation'
V1_ADAPTER_DIR  = f'{PROJECT_DIR}/adapters/mi-therapy-gemma2-v1'
V2_ADAPTER_DIR  = f'{PROJECT_DIR}/adapters/mi-therapy-gemma2-v2'
MODEL_ID        = 'google/gemma-2-2b-it'

# Verify eval scenarios exist
assert os.path.exists(EVAL_SCENARIOS), f'Upload eval_scenarios.jsonl to: {EVAL_SCENARIOS}'
assert os.path.exists(V1_ADAPTER_DIR), f'V1 adapter not found at: {V1_ADAPTER_DIR}'
assert os.path.exists(V2_ADAPTER_DIR), f'V2 adapter not found at: {V2_ADAPTER_DIR}'

# Load scenarios
scenarios = []
with open(EVAL_SCENARIOS, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            scenarios.append(json.loads(line))

print(f'Loaded {len(scenarios)} eval scenarios')
print(f'V1 adapter: {V1_ADAPTER_DIR}')
print(f'V2 adapter: {V2_ADAPTER_DIR}')

In [ ]:
# ============================================================
# CELL 3: Shared helpers (run once)
# ============================================================
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

SYSTEM_PROMPT = """You are a compassionate, skilled therapist specializing in Motivational Interviewing (MI) for alcohol-related concerns.

Your approach:
- Use OARS: Open questions, Affirmations, Reflections, Summaries
- Meet the client where they are — never confront or lecture
- Roll with resistance, don't fight it
- Evoke the client's own motivation for change
- Respect autonomy — the client decides if and when they change
- Reflect emotions and defensiveness back with empathy

Respond with a single, thoughtful therapist response (2-4 sentences). Do NOT include any labels, headers, or metadata — just the therapist's words."""


def build_prompt(scenario: dict, tokenizer) -> torch.Tensor:
    """Build tokenized input for a scenario using chat template."""
    history_parts = []
    for msg in scenario.get('conversation_history', []):
        role = 'Therapist' if msg['role'] == 'assistant' else 'Client'
        history_parts.append(f"{role}: {msg['content']}")

    history_str = '\n'.join(history_parts)
    if history_str:
        user_content = f"{SYSTEM_PROMPT}\n\n## Conversation so far:\n{history_str}\n\nClient: {scenario['client_message']}\nNow respond as the therapist. Only output the therapist's response, nothing else:"
    else:
        user_content = f"{SYSTEM_PROMPT}\n\nClient: {scenario['client_message']}\nNow respond as the therapist. Only output the therapist's response, nothing else:"

    messages = [{'role': 'user', 'content': user_content}]
    encoded = tokenizer.apply_chat_template(
        messages,
        return_tensors='pt',
        add_generation_prompt=True,
    )
    # Newer transformers returns BatchEncoding, older returns raw tensor
    if hasattr(encoded, 'input_ids'):
        input_ids = encoded.input_ids
    else:
        input_ids = encoded
    return input_ids.to('cuda')


def generate_response(model, tokenizer, scenario: dict) -> str:
    """Generate a single therapist response."""
    input_ids = build_prompt(scenario, tokenizer)
    input_len = input_ids.shape[1]

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens
    new_tokens = output[0][input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response


def run_inference(model, tokenizer, scenarios: list, config_name: str):
    """Run inference on all scenarios and save results."""
    output_file = f'{OUTPUT_DIR}/model_responses_{config_name}.jsonl'

    # Load existing to support resuming
    existing = {}
    if os.path.exists(output_file):
        with open(output_file, encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    item = json.loads(line)
                    existing[item['scenario']['id']] = item
        print(f'  Resuming: {len(existing)} already done, skipping those')

    model.eval()
    model.config.use_cache = True

    generated = 0
    skipped = 0
    failed = 0

    with open(output_file, 'a', encoding='utf-8') as f:
        for i, scenario in enumerate(scenarios):
            if scenario['id'] in existing:
                skipped += 1
                continue

            try:
                response = generate_response(model, tokenizer, scenario)

                if not response or len(response) < 10:
                    print(f'  [{i+1}/{len(scenarios)}] SHORT RESPONSE: {scenario["id"]}')
                    response = '[GENERATION FAILED]'
                    failed += 1
                else:
                    generated += 1

                result = {
                    'scenario': scenario,
                    'model_response': response,
                    'config': config_name,
                }
                f.write(json.dumps(result, ensure_ascii=False) + '\n')
                f.flush()

                if (i + 1) % 20 == 0:
                    print(f'  [{i+1}/{len(scenarios)}] Generated: {generated} | Skipped: {skipped} | Failed: {failed}')
                    print(f'  Sample: {response[:100]}...')

            except Exception as e:
                import traceback
                print(f'  [{i+1}/{len(scenarios)}] ERROR: {scenario["id"]} — {type(e).__name__}: {str(e)[:150]}')
                traceback.print_exc()
                failed += 1

    print(f'\nDone with {config_name}!')
    print(f'  Generated: {generated} | Skipped: {skipped} | Failed: {failed}')
    print(f'  Saved to: {output_file}')


def load_base_model():
    """Load Gemma 2 2B IT in 4-bit."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
    )
    print(f'GPU memory after load: {torch.cuda.memory_allocated()/1024**3:.2f} GB')
    return model, tokenizer


def free_memory(model):
    """Free GPU memory between configs."""
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f'GPU memory after cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB')


print('Helpers ready!')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB total')

In [ ]:
# ============================================================
# CELL 4: CONFIG = BASE (Gemma 2 2B IT, no fine-tuning)
# Estimated time: ~25-35 minutes on T4
# ============================================================
print('=' * 60)
print('RUNNING: base (Gemma 2 2B IT — no fine-tuning)')
print('=' * 60)

model_base, tokenizer = load_base_model()
run_inference(model_base, tokenizer, scenarios, config_name='base')
free_memory(model_base)

In [ ]:
# ============================================================
# CELL 5: CONFIG = V1 (fine-tuned on 95 examples)
# Estimated time: ~25-35 minutes on T4
# ============================================================
print('=' * 60)
print('RUNNING: v1 (fine-tuned, 95 examples)')
print('=' * 60)

# Load base model then attach v1 adapter
model_base_v1, tokenizer = load_base_model()
model_v1 = PeftModel.from_pretrained(model_base_v1, V1_ADAPTER_DIR)
print(f'V1 adapter loaded from: {V1_ADAPTER_DIR}')

run_inference(model_v1, tokenizer, scenarios, config_name='v1')
free_memory(model_v1)

In [ ]:
# ============================================================
# CELL 6: CONFIG = V2 (fine-tuned on 363 examples)
# Estimated time: ~25-35 minutes on T4
# ============================================================
print('=' * 60)
print('RUNNING: v2 (fine-tuned, 363 examples)')
print('=' * 60)

# Load base model then attach v2 adapter
model_base_v2, tokenizer = load_base_model()
model_v2 = PeftModel.from_pretrained(model_base_v2, V2_ADAPTER_DIR)
print(f'V2 adapter loaded from: {V2_ADAPTER_DIR}')

run_inference(model_v2, tokenizer, scenarios, config_name='v2')
free_memory(model_v2)

In [ ]:
# ============================================================
# CELL 7: Verify all outputs
# ============================================================
print('Verifying outputs...')
print()

for config in ['base', 'v1', 'v2']:
    path = f'{OUTPUT_DIR}/model_responses_{config}.jsonl'
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            lines = [l for l in f if l.strip()]
        failed = sum(1 for l in lines if '[GENERATION FAILED]' in l or '[CLI' in l)
        print(f'  {config}: {len(lines)} responses ({failed} failed)')

        # Show a sample response
        sample = json.loads(lines[0])
        print(f'  Sample ({config}): {sample["model_response"][:120]}...')
        print()
    else:
        print(f'  {config}: NOT FOUND at {path}')

print('All done! Download these files to your local machine:')
for config in ['base', 'v1', 'v2']:
    print(f'  {OUTPUT_DIR}/model_responses_{config}.jsonl')
print()
print('Then run locally:')
print('  python scripts/score_with_claude.py --config base')
print('  python scripts/score_with_claude.py --config v1')
print('  python scripts/score_with_claude.py --config v2')
print('  python scripts/score_with_claude.py --compare')

In [ ]:
# ============================================================
# CELL 7b: CONFIG = V2+RAG (Enhanced v2 — fine-tuned v2 + enriched RAG)
# Upload eval_scenarios_rag.jsonl to Drive FIRST
# Estimated time: ~35-40 minutes on T4
# ============================================================
import json, os

RAG_SCENARIOS_FILE = f'{PROJECT_DIR}/data/evaluation/eval_scenarios_rag.jsonl'
assert os.path.exists(RAG_SCENARIOS_FILE), f'Upload eval_scenarios_rag.jsonl to: {RAG_SCENARIOS_FILE}'

# Load RAG-enriched scenarios
rag_scenarios = []
with open(RAG_SCENARIOS_FILE, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rag_scenarios.append(json.loads(line))
print(f'Loaded {len(rag_scenarios)} RAG-enriched scenarios')

# Verify RAG context
sample = rag_scenarios[0]
print(f'RAG chunks for {sample["id"]}: {len(sample.get("rag_context", []))} chunks')
for chunk in sample.get('rag_context', [])[:2]:
    print(f'  [{chunk["source"]}] {chunk["content"][:80]}...')


def build_rag_prompt(scenario: dict, tokenizer):
    """
    Build prompt with structured RAG context injection.
    
    Format: Expert reference examples FIRST, then conversation, then client message.
    This puts the MI knowledge closest to the system prompt, giving the model
    a "cheat sheet" before it sees what to respond to.
    """
    rag_chunks = scenario.get('rag_context', [])

    # Separate example-based chunks from guideline chunks
    examples = []
    guidelines = []
    for c in rag_chunks:
        if 'Example' in c['content'] or 'Client' in c['content']:
            examples.append(c)
        else:
            guidelines.append(c)

    rag_section = ''
    if guidelines:
        guideline_text = '\n'.join([c['content'] for c in guidelines[:2]])
        rag_section += f"\n\n## MI Guidelines for this situation:\n{guideline_text}"

    if examples:
        example_text = '\n\n'.join([c['content'] for c in examples[:3]])
        rag_section += f"\n\n## How expert MI therapists handled similar situations:\n{example_text}"

    # Build conversation history
    history_parts = []
    for msg in scenario.get('conversation_history', []):
        role = 'Therapist' if msg['role'] == 'assistant' else 'Client'
        history_parts.append(f"{role}: {msg['content']}")

    history_str = '\n'.join(history_parts)
    if history_str:
        user_content = f"{SYSTEM_PROMPT}{rag_section}\n\n## Conversation so far:\n{history_str}\n\nClient: {scenario['client_message']}\nNow respond as the therapist. Use the MI techniques shown above. Only output the therapist's words:"
    else:
        user_content = f"{SYSTEM_PROMPT}{rag_section}\n\nClient: {scenario['client_message']}\nNow respond as the therapist. Use the MI techniques shown above. Only output the therapist's words:"

    messages = [{'role': 'user', 'content': user_content}]
    encoded = tokenizer.apply_chat_template(
        messages,
        return_tensors='pt',
        add_generation_prompt=True,
    )
    if hasattr(encoded, 'input_ids'):
        input_ids = encoded.input_ids
    else:
        input_ids = encoded
    return input_ids.to('cuda')


def generate_rag_response(model, tokenizer, scenario: dict) -> str:
    """Generate response using RAG-augmented prompt."""
    input_ids = build_rag_prompt(scenario, tokenizer)
    input_len = input_ids.shape[1]

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response


print('\nLoading V2 model for RAG inference...')
print('=' * 60)
print('RUNNING: v2_rag (fine-tuned v2 + enriched RAG knowledge base)')
print(f'  Knowledge: 19 files, 335 KB, ~1000+ chunks')
print(f'  Retrieval: multi-query MMR with 5 chunks per scenario')
print('=' * 60)

model_base_rag, tokenizer = load_base_model()
model_v2_rag = PeftModel.from_pretrained(model_base_rag, V2_ADAPTER_DIR)
model_v2_rag.eval()
model_v2_rag.config.use_cache = True
print(f'V2 adapter loaded from: {V2_ADAPTER_DIR}')

# Delete old output to start fresh
output_file = f'{OUTPUT_DIR}/model_responses_v2_rag.jsonl'
if os.path.exists(output_file):
    os.remove(output_file)
    print('Deleted old v2_rag responses — starting fresh with enhanced RAG')

generated = skipped = failed = 0

with open(output_file, 'a', encoding='utf-8') as f:
    for i, scenario in enumerate(rag_scenarios):
        try:
            response = generate_rag_response(model_v2_rag, tokenizer, scenario)
            if not response or len(response) < 10:
                response = '[GENERATION FAILED]'
                failed += 1
            else:
                generated += 1

            result = {
                'scenario': scenario,
                'model_response': response,
                'config': 'v2_rag',
            }
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
            f.flush()

            if (i + 1) % 20 == 0:
                print(f'  [{i+1}/{len(rag_scenarios)}] Generated: {generated} | Skipped: {skipped} | Failed: {failed}')
                print(f'  Sample: {response[:100]}...')

        except Exception as e:
            import traceback
            print(f'  [{i+1}/{len(rag_scenarios)}] ERROR: {scenario["id"]} — {type(e).__name__}: {str(e)[:150]}')
            traceback.print_exc()
            failed += 1

free_memory(model_v2_rag)

print(f'\nDone! v2_rag: {generated} generated | {skipped} skipped | {failed} failed')
print(f'Download: {output_file}')
print('Then run locally: python scripts/score_with_claude.py --config v2_rag')

In [ ]:
# ============================================================
# CELL 7c: CONFIG = BASE+RAG (no fine-tuning, just RAG context)
# This isolates RAG's contribution without fine-tuning
# Upload eval_scenarios_rag.jsonl to Drive FIRST
# Estimated time: ~35 minutes on T4
# ============================================================
import json, os

RAG_SCENARIOS_FILE = f'{PROJECT_DIR}/data/evaluation/eval_scenarios_rag.jsonl'
assert os.path.exists(RAG_SCENARIOS_FILE), f'Upload eval_scenarios_rag.jsonl to: {RAG_SCENARIOS_FILE}'

# Load RAG-enriched scenarios
rag_scenarios = []
with open(RAG_SCENARIOS_FILE, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rag_scenarios.append(json.loads(line))
print(f'Loaded {len(rag_scenarios)} RAG-enriched scenarios')


def build_rag_prompt(scenario: dict, tokenizer):
    """Build prompt with structured RAG context injection."""
    rag_chunks = scenario.get('rag_context', [])

    examples = []
    guidelines = []
    for c in rag_chunks:
        if 'Example' in c['content'] or 'Client' in c['content']:
            examples.append(c)
        else:
            guidelines.append(c)

    rag_section = ''
    if guidelines:
        guideline_text = '\n'.join([c['content'] for c in guidelines[:2]])
        rag_section += f"\n\n## MI Guidelines for this situation:\n{guideline_text}"

    if examples:
        example_text = '\n\n'.join([c['content'] for c in examples[:3]])
        rag_section += f"\n\n## How expert MI therapists handled similar situations:\n{example_text}"

    history_parts = []
    for msg in scenario.get('conversation_history', []):
        role = 'Therapist' if msg['role'] == 'assistant' else 'Client'
        history_parts.append(f"{role}: {msg['content']}")

    history_str = '\n'.join(history_parts)
    if history_str:
        user_content = f"{SYSTEM_PROMPT}{rag_section}\n\n## Conversation so far:\n{history_str}\n\nClient: {scenario['client_message']}\nNow respond as the therapist. Use the MI techniques shown above. Only output the therapist's words:"
    else:
        user_content = f"{SYSTEM_PROMPT}{rag_section}\n\nClient: {scenario['client_message']}\nNow respond as the therapist. Use the MI techniques shown above. Only output the therapist's words:"

    messages = [{'role': 'user', 'content': user_content}]
    encoded = tokenizer.apply_chat_template(
        messages,
        return_tensors='pt',
        add_generation_prompt=True,
    )
    if hasattr(encoded, 'input_ids'):
        input_ids = encoded.input_ids
    else:
        input_ids = encoded
    return input_ids.to('cuda')


def generate_rag_response(model, tokenizer, scenario: dict) -> str:
    """Generate response using RAG-augmented prompt."""
    input_ids = build_rag_prompt(scenario, tokenizer)
    input_len = input_ids.shape[1]

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response


print('=' * 60)
print('RUNNING: base_rag (NO fine-tuning, just RAG)')
print('This isolates RAG contribution for the 2x2 factorial analysis')
print('=' * 60)

# Load BASE model — NO adapter
model_base_rag, tokenizer = load_base_model()
model_base_rag.eval()
model_base_rag.config.use_cache = True
print('Base model loaded (NO adapter)')

output_file = f'{OUTPUT_DIR}/model_responses_base_rag.jsonl'
existing = {}
if os.path.exists(output_file):
    with open(output_file, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                existing[item['scenario']['id']] = item
    print(f'Resuming: {len(existing)} already done')

generated = skipped = failed = 0

with open(output_file, 'a', encoding='utf-8') as f:
    for i, scenario in enumerate(rag_scenarios):
        if scenario['id'] in existing:
            skipped += 1
            continue
        try:
            response = generate_rag_response(model_base_rag, tokenizer, scenario)
            if not response or len(response) < 10:
                response = '[GENERATION FAILED]'
                failed += 1
            else:
                generated += 1

            result = {
                'scenario': scenario,
                'model_response': response,
                'config': 'base_rag',
            }
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
            f.flush()

            if (i + 1) % 20 == 0:
                print(f'  [{i+1}/{len(rag_scenarios)}] Generated: {generated} | Skipped: {skipped} | Failed: {failed}')
                print(f'  Sample: {response[:100]}...')

        except Exception as e:
            import traceback
            print(f'  [{i+1}/{len(rag_scenarios)}] ERROR: {scenario["id"]} — {type(e).__name__}: {str(e)[:150]}')
            traceback.print_exc()
            failed += 1

free_memory(model_base_rag)

print(f'\nDone! base_rag: {generated} generated | {skipped} skipped | {failed} failed')
print(f'Download: {output_file}')
print('Then run locally: python scripts/score_with_claude.py --config base_rag')